# SGD Logistic Baseline: MFCC-10
Trains probability-based linear multilabel and binary UUV baselines for normal, M-filtered, and W-filtered MFCC-10 data. Use a High-RAM CPU runtime.

In [2]:
import sys
from pathlib import Path

REPO_RAW_BASE_URL = "https://raw.githubusercontent.com/Nabuhodonozzor/uuv-detection/main"
COMMON_UTILS_FILE = "common_utils.py"
MODEL_UTILS_FILE = "sgd_utils.py"
MODEL_DIR = "Baselines"
common_dirs = [Path.cwd() / "utils", Path.cwd().parent / "utils", Path("/content/utils"), Path("/content/drive/MyDrive/STUDA/src/utils")]
model_dirs = [Path.cwd(), Path.cwd() / MODEL_DIR, Path.cwd().parent / MODEL_DIR, Path("/content") / MODEL_DIR, Path("/content/drive/MyDrive/STUDA/src") / MODEL_DIR]
common_dir = next((directory for directory in common_dirs if (directory / COMMON_UTILS_FILE).exists()), None)
model_dir = next((directory for directory in model_dirs if (directory / MODEL_UTILS_FILE).exists()), None)

if common_dir is None or model_dir is None:
    import urllib.request
    common_dir = Path("/content/utils")
    model_dir = Path("/content") / MODEL_DIR
    common_dir.mkdir(parents=True, exist_ok=True)
    model_dir.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(f"{REPO_RAW_BASE_URL}/utils/{COMMON_UTILS_FILE}", common_dir / COMMON_UTILS_FILE)
    urllib.request.urlretrieve(f"{REPO_RAW_BASE_URL}/{MODEL_DIR}/{MODEL_UTILS_FILE}", model_dir / MODEL_UTILS_FILE)

sys.path.insert(0, str(common_dir))
sys.path.insert(0, str(model_dir))
print(f"Using common utilities from: {common_dir}")
print(f"Using baseline utilities from: {model_dir}")


Using common utilities from: /content/utils
Using baseline utilities from: /content/Baselines


In [3]:
import pandas as pd
from IPython.display import display
from google.colab import files

from common_utils import configure_kaggle_access, evaluate_models_for_variants, extract_zip, prepare_mfcc_dataset_variants, zip_artifacts
from sgd_utils import build_sgd_models_for_variants, save_sgd_artifacts, train_sgd_models_for_variants


In [4]:
DATASET_KEY = "mfcc10"
DATASET_LABEL = "MFCC-10"
DATASET_SLUG = "pawedyrda/mfcc10"
ARCHIVE_PATH = Path("/content/mfcc10.zip")


In [5]:
configure_kaggle_access("Kaggle")
if not ARCHIVE_PATH.exists():
    !kaggle datasets download -d {DATASET_SLUG} -p /content
else:
    print(f"Reusing downloaded archive: {ARCHIVE_PATH}")
DATA_PATH = extract_zip(ARCHIVE_PATH, "/content")
print(f"Dataset extracted to: {DATA_PATH}")


Dataset URL: https://www.kaggle.com/datasets/pawedyrda/mfcc10
License(s): unknown
100% 218M/218M [00:05<00:00, 43.3MB/s] 

Dataset extracted to: /content/mfcc10


In [6]:
variants = prepare_mfcc_dataset_variants(DATA_PATH)
print("Normal train shape:", variants.normal.train_data.shape)
print("M train shape:", variants.m.train_data.shape)
print("W train shape:", variants.w.train_data.shape)


Scanning feature data in: /content/mfcc10
Found .npz files: 9527, skipped: 0
Loaded samples: 9527
X shape: (9527, 618, 10)
y shape: (9527, 17)
Scanning feature data in: /content/mfcc10 (UUV filter: M)
Found .npz files: 9527, skipped: 523
Loaded samples: 9004
X shape: (9004, 618, 10)
y shape: (9004, 17)
Scanning feature data in: /content/mfcc10 (UUV filter: W)
Found .npz files: 9527, skipped: 379
Loaded samples: 9148
X shape: (9148, 618, 10)
y shape: (9148, 17)
Normal train shape: (6096, 618, 10)
M train shape: (5762, 618, 10)
W train shape: (5854, 618, 10)


In [7]:
multilabel_models = build_sgd_models_for_variants(variants, model_type="multilabel")
train_sgd_models_for_variants(multilabel_models, variants, model_type="multilabel")

binary_models = build_sgd_models_for_variants(variants, model_type="binary")
train_sgd_models_for_variants(binary_models, variants, model_type="binary")


Training multilabel SGD logistic model for variant: normal
Training multilabel SGD logistic model for variant: M
Training multilabel SGD logistic model for variant: W
Training binary SGD logistic model for variant: normal
Training binary SGD logistic model for variant: M
Training binary SGD logistic model for variant: W


{'normal': Pipeline(steps=[('flatten',
                  FunctionTransformer(func=<function flatten_mfcc_features at 0x7e39fb8220c0>)),
                 ('scale', StandardScaler()),
                 ('classifier',
                  SGDClassifier(average=True, class_weight='balanced',
                                early_stopping=True, loss='log_loss',
                                random_state=42))]),
 'M': Pipeline(steps=[('flatten',
                  FunctionTransformer(func=<function flatten_mfcc_features at 0x7e39fb8220c0>)),
                 ('scale', StandardScaler()),
                 ('classifier',
                  SGDClassifier(average=True, class_weight='balanced',
                                early_stopping=True, loss='log_loss',
                                random_state=42))]),
 'W': Pipeline(steps=[('flatten',
                  FunctionTransformer(func=<function flatten_mfcc_features at 0x7e39fb8220c0>)),
                 ('scale', StandardScaler()),
            

In [8]:
multilabel_results = evaluate_models_for_variants(multilabel_models, variants, "multilabel", DATASET_LABEL)
binary_results = evaluate_models_for_variants(binary_models, variants, "binary", DATASET_LABEL)
comparison_results = pd.concat([multilabel_results.assign(task="multilabel"), binary_results.assign(task="binary")], ignore_index=True)
display(comparison_results[["task", "Model", "precision", "recall", "f1-score", "support"]])



--- Evaluation for: Multilabel normal MFCC-10 ---

                   precision    recall  f1-score   support

ArtificialSignals       0.28      0.86      0.43       201
 BigPassengerShip       0.06      1.00      0.11        36
            Cargo       0.07      0.93      0.12        46
         FishBoat       0.08      0.92      0.14        48
        GreenCity       0.11      0.74      0.19        89
           KaiYan       0.03      1.00      0.05        20
          KaiYuan       0.24      0.68      0.36       250
        MotorBoat       0.04      1.00      0.08        29
              No7       0.15      0.77      0.25       120
       PoliceBoat       0.02      1.00      0.04        14
          QianDao       0.62      0.78      0.69       658
        SpeedBoat       0.62      0.63      0.62       879
          TheEarl       0.18      0.99      0.30       109
        TheKnight       0.09      0.98      0.16        55
              UUV       0.30      0.88      0.45       195
   

,task,Model,precision,recall,f1-score,support
0,multilabel,Multilabel normal MFCC-10,0.300000,0.876923,0.447059,195.0
1,multilabel,Multilabel M MFCC-10,0.116519,0.908046,0.206536,87.0
2,multilabel,Multilabel W MFCC-10,0.175725,0.950980,0.296636,102.0
3,binary,Binary normal MFCC-10,0.300000,0.876923,0.447059,195.0
4,binary,Binary M MFCC-10,0.116519,0.908046,0.206536,87.0
5,binary,Binary W MFCC-10,0.175725,0.950980,0.296636,102.0


In [9]:
save_dir = save_sgd_artifacts(f"/content/saved_artifacts/sgd_{DATASET_KEY}", DATASET_KEY, multilabel_models, binary_models, multilabel_results, binary_results)
comparison_results.to_csv(save_dir / f"sgd_comparison_{DATASET_KEY}.csv", index=False)
archive_path = zip_artifacts(save_dir, f"/content/sgd_models_and_results_{DATASET_KEY}.zip")
files.download(str(archive_path))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>